# Crawler BRACIS (Essencial)

Coleta trabalhos das edicoes de 2023, 2024 e 2025 do BRACIS no SOL/SBC e consolida em `DataFrame`.

In [1]:
import re
import time
from dataclasses import dataclass
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://sol.sbc.org.br/index.php"
BRACIS_SLUG = "bracis"
ANOS_ALVO = {2023, 2024, 2025}
REQUEST_TIMEOUT = 30
SLEEP_SECONDS = 0.2
USER_AGENT = (
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
)


In [2]:
@dataclass(frozen=True)
class Edicao:
    ano: int
    titulo: str
    link: str


def baixar_soup(url: str, session: requests.Session) -> BeautifulSoup:
    resposta = session.get(url, timeout=REQUEST_TIMEOUT)
    resposta.raise_for_status()
    return BeautifulSoup(resposta.text, "html.parser")


def extrair_ano(texto: str):
    if not texto:
        return None
    match = re.search(r"\b(20\d{2})\b", texto)
    return int(match.group(1)) if match else None


def listar_edicoes_bracis(session: requests.Session, anos_alvo):
    archive_url = f"{BASE_URL}/{BRACIS_SLUG}/issue/archive"
    soup = baixar_soup(archive_url, session)
    edicoes = []

    for bloco in soup.select("div.obj_issue_summary"):
        tag_titulo = bloco.select_one("a.title")
        if not tag_titulo:
            continue

        titulo_edicao = tag_titulo.get_text(" ", strip=True)
        ano_edicao = extrair_ano(titulo_edicao)

        if ano_edicao not in anos_alvo:
            continue

        link_edicao = urljoin(archive_url, tag_titulo.get("href", "").strip())
        edicoes.append(Edicao(ano=ano_edicao, titulo=titulo_edicao, link=link_edicao))

    edicoes.sort(key=lambda item: (item.ano, item.titulo), reverse=True)
    return edicoes


def listar_artigos_edicao(url_edicao: str, session: requests.Session):
    soup = baixar_soup(url_edicao, session)
    artigos = []
    links_vistos = set()

    for bloco in soup.select("div.obj_article_summary"):
        tag_artigo = bloco.select_one("div.title a")
        if not tag_artigo:
            continue

        link_artigo = urljoin(url_edicao, tag_artigo.get("href", "").strip())
        if not link_artigo or link_artigo in links_vistos:
            continue

        links_vistos.add(link_artigo)
        artigos.append(
            {
                "titulo_edicao": tag_artigo.get_text(" ", strip=True),
                "link_artigo": link_artigo,
            }
        )

    return artigos


def extrair_titulo_resumo(url_artigo: str, titulo_fallback: str, session: requests.Session):
    soup = baixar_soup(url_artigo, session)

    tag_titulo = soup.select_one("h1.page_title")
    titulo = tag_titulo.get_text(" ", strip=True) if tag_titulo else titulo_fallback

    resumo = ""
    tag_resumo = soup.select_one("div.item.abstract")
    if tag_resumo:
        tag_label = tag_resumo.select_one("h3.label")
        if tag_label:
            tag_label.decompose()
        resumo = tag_resumo.get_text(" ", strip=True)

    return titulo, resumo


def coletar_trabalhos_bracis(anos_alvo=ANOS_ALVO, sleep_seconds=SLEEP_SECONDS):
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})

    registros = []
    edicoes = listar_edicoes_bracis(session, anos_alvo)

    print(f"Edicoes BRACIS encontradas para {sorted(anos_alvo)}: {len(edicoes)}")

    for edicao in edicoes:
        print(f"- {edicao.ano}: {edicao.titulo}")
        artigos = listar_artigos_edicao(edicao.link, session)
        print(f"  Artigos na edicao: {len(artigos)}")

        total_artigos = len(artigos)
        for indice, artigo in enumerate(artigos, start=1):
            time.sleep(sleep_seconds)
            try:
                titulo, resumo = extrair_titulo_resumo(
                    artigo["link_artigo"], artigo["titulo_edicao"], session
                )
            except requests.RequestException as erro:
                print(f"  [aviso] erro em {artigo['link_artigo']}: {erro}")
                continue

            registros.append(
                {
                    "Conferencia": "BRACIS",
                    "Ano": edicao.ano,
                    "Edicao": edicao.titulo,
                    "Titulo": titulo,
                    "Resumo": resumo,
                    "Link": artigo["link_artigo"],
                }
            )

            if indice % 25 == 0 or indice == total_artigos:
                print(f"  Progresso: {indice}/{total_artigos}")

    df = pd.DataFrame(
        registros,
        columns=["Conferencia", "Ano", "Edicao", "Titulo", "Resumo", "Link"],
    )

    if not df.empty:
        df = (
            df.drop_duplicates(subset=["Ano", "Titulo", "Link"])
            .sort_values(["Ano", "Titulo"], ascending=[False, True])
            .reset_index(drop=True)
        )

    return df


In [3]:
df_bracis = coletar_trabalhos_bracis(anos_alvo={2023, 2024, 2025}, sleep_seconds=0.2)
df_bracis.head(10)

Edicoes BRACIS encontradas para [2023, 2024, 2025]: 3
- 2025: 2025: Anais da XXXV Brazilian Conference on Intelligent Systems
  Artigos na edicao: 148
  Progresso: 25/148
  Progresso: 50/148
  Progresso: 75/148
  Progresso: 100/148
  Progresso: 125/148
  Progresso: 148/148
- 2024: 2024: Anais da XXXIV Brazilian Conference on Intelligent Systems
  Artigos na edicao: 116
  Progresso: 25/116
  Progresso: 50/116
  Progresso: 75/116
  Progresso: 100/116
  Progresso: 116/116
- 2023: 2023: Anais da XII Brazilian Conference on Intelligent Systems
  Artigos na edicao: 90
  Progresso: 25/90
  Progresso: 50/90
  [aviso] erro em https://sol.sbc.org.br/index.php/bracis/article/view/28414: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
  Progresso: 75/90
  Progresso: 90/90


,Conferencia,Ano,Edicao,Titulo,Resumo,Link
0,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Categorical Kalman Filter for Human Activity...,Human Activity Recognition (HAR) has gained in...,https://sol.sbc.org.br/index.php/bracis/articl...
1,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Comparative Study of Machine Learning Models...,The bee flora refers to the group of plants fr...,https://sol.sbc.org.br/index.php/bracis/articl...
2,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Comprehensive Study of Fitness Landscapes in...,The search space in Automated Machine Learning...,https://sol.sbc.org.br/index.php/bracis/articl...
3,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Double Transfer Learning Approach for Improv...,Cancer is one of the most devastating health c...,https://sol.sbc.org.br/index.php/bracis/articl...
4,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Fully Automatic Approach for COVID-19 Diagno...,The pandemic caused by the COVID-19 virus has ...,https://sol.sbc.org.br/index.php/bracis/articl...
5,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Generative Domain Adaptation Scheme for Swif...,Deep learning models have demonstrated remarka...,https://sol.sbc.org.br/index.php/bracis/articl...
6,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Livestock Monitoring System and Machine Lear...,Timely disease diagnosis is essential for redu...,https://sol.sbc.org.br/index.php/bracis/articl...
7,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Multi-Dimensional Comparative Study of Gener...,Synthetic data is increasingly important in pr...,https://sol.sbc.org.br/index.php/bracis/articl...
8,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Multidimensional Analysis of Swarm Dynamics ...,Traditional assessments of metaheuristics typi...,https://sol.sbc.org.br/index.php/bracis/articl...
9,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A NAS-Optimized Deep Learning Model for Custom...,Deep Learning (DL) methods offer competitive a...,https://sol.sbc.org.br/index.php/bracis/articl...


In [4]:
print(f"Total de registros: {len(df_bracis)}")
print("\nColunas:", list(df_bracis.columns))
print("\nTrabalhos por ano:")
print(df_bracis["Ano"].value_counts().sort_index(ascending=False))

Total de registros: 353

Colunas: ['Conferencia', 'Ano', 'Edicao', 'Titulo', 'Resumo', 'Link']

Trabalhos por ano:
Ano
2025    148
2024    116
2023     89
Name: count, dtype: int64


In [5]:
arquivo_saida = "bracis_trabalhos_2023_2025.csv"
df_bracis.to_csv(arquivo_saida, index=False)
print(f"CSV salvo em: {arquivo_saida}")

CSV salvo em: bracis_trabalhos_2023_2025.csv


## Parte 1 - NMF (metodo escolhido) com BoW e TF-IDF

Fluxo desta parte:

- preparar o corpus textual (`Titulo + Resumo`);
- gerar representacoes BoW e TF-IDF;
- aplicar NMF para extrair topicos;
- interpretar os topicos por palavras-chave, volume e exemplos.


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import NMF

pd.set_option("display.max_colwidth", 220)


In [ ]:
# Usa o dataframe coletado na etapa anterior do notebook.
df_nlp = df_bracis.copy()

# Limpeza minima para padronizar espacos e evitar nulos no texto.
for coluna in ["Titulo", "Resumo"]:
    df_nlp[coluna] = (
        df_nlp[coluna]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# Documento final = titulo + resumo.
df_nlp["texto"] = (df_nlp["Titulo"] + ". " + df_nlp["Resumo"]).str.strip()
df_nlp = df_nlp[df_nlp["texto"].str.len() > 0].reset_index(drop=True)

print(f"Documentos no corpus: {len(df_nlp)}")
df_nlp[["Ano", "Titulo", "Resumo"]].head(3)


In [ ]:
# (a) Bag of Words
bow_vectorizer = CountVectorizer(
    stop_words="english",
    min_df=3,
    max_df=0.9,
    ngram_range=(1, 2),
)

X_bow = bow_vectorizer.fit_transform(df_nlp["texto"])
vocab_bow = bow_vectorizer.get_feature_names_out()

# Soma por coluna para obter termos mais frequentes no corpus.
freq_bow = np.asarray(X_bow.sum(axis=0)).ravel()
idx_top_bow = freq_bow.argsort()[::-1][:25]
bow_top_terms = pd.DataFrame(
    {
        "termo": vocab_bow[idx_top_bow],
        "frequencia": freq_bow[idx_top_bow],
    }
)

print("Matriz BoW:", X_bow.shape)
print("Vocabulario BoW:", len(vocab_bow))
bow_top_terms


In [ ]:
# (a) TF-IDF
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    min_df=3,
    max_df=0.9,
    ngram_range=(1, 2),
    sublinear_tf=True,
)

X_tfidf = tfidf_vectorizer.fit_transform(df_nlp["texto"])
vocab_tfidf = tfidf_vectorizer.get_feature_names_out()

# Media do peso TF-IDF por termo para destacar termos informativos.
mean_tfidf = np.asarray(X_tfidf.mean(axis=0)).ravel()
idx_top_tfidf = mean_tfidf.argsort()[::-1][:25]
tfidf_top_terms = pd.DataFrame(
    {
        "termo": vocab_tfidf[idx_top_tfidf],
        "peso_medio_tfidf": mean_tfidf[idx_top_tfidf],
    }
)

print("Matriz TF-IDF:", X_tfidf.shape)
print("Vocabulario TF-IDF:", len(vocab_tfidf))
tfidf_top_terms


In [ ]:
def top_words_by_topic(components, feature_names, n_top_words=12):
    """Retorna uma tabela com as palavras mais relevantes de cada topico."""
    linhas = []
    for topic_id, topic_weights in enumerate(components):
        top_idx = topic_weights.argsort()[::-1][:n_top_words]
        palavras = [feature_names[i] for i in top_idx]
        linhas.append({"topico": topic_id, "palavras_chave": ", ".join(palavras)})
    return pd.DataFrame(linhas)


In [ ]:
# (b) NMF sobre TF-IDF (metodo escolhido)
N_TOPICS = 8
N_TOP_WORDS = 12

nmf_model = NMF(
    n_components=N_TOPICS,
    random_state=42,
    init="nndsvda",
    max_iter=600,
)

# W = documento x topico | H = topico x termo
W_nmf = nmf_model.fit_transform(X_tfidf)
H_nmf = nmf_model.components_

nmf_topics_df = top_words_by_topic(H_nmf, vocab_tfidf, n_top_words=N_TOP_WORDS)
df_nlp["topico_nmf"] = W_nmf.argmax(axis=1)
df_nlp["score_topico_nmf"] = W_nmf.max(axis=1)

print("Erro de reconstrucao (NMF):", round(nmf_model.reconstruction_err_, 4))
nmf_topics_df


In [ ]:
# (c) Interpretacao dos topicos (NMF)
def interpretar_topicos_nmf(df_docs, topicos_df, top_n_titulos=3):
    linhas = []
    for _, linha in topicos_df.iterrows():
        t = int(linha["topico"])
        sub = df_docs[df_docs["topico_nmf"] == t].copy()

        qtd_docs = len(sub)
        por_ano = sub["Ano"].value_counts().sort_index().to_dict()
        # Titulos com maior contribuicao ao topico.
        titulos_rep = (
            sub.sort_values("score_topico_nmf", ascending=False)["Titulo"]
            .head(top_n_titulos)
            .tolist()
        )

        termos = linha["palavras_chave"].split(", ")[:4]
        interpretacao = (
            f"Topico {t} foca em {', '.join(termos)}. "
            f"Documentos: {qtd_docs}. Distribuicao por ano: {por_ano}."
        )

        linhas.append(
            {
                "topico": t,
                "palavras_chave": linha["palavras_chave"],
                "qtd_documentos": qtd_docs,
                "distribuicao_anos": por_ano,
                "titulos_representativos": " | ".join(titulos_rep),
                "interpretacao_inicial": interpretacao,
            }
        )

    return pd.DataFrame(linhas).sort_values("qtd_documentos", ascending=False)

interpretacao_nmf_df = interpretar_topicos_nmf(df_nlp, nmf_topics_df)
interpretacao_nmf_df


### Texto para relatorio (NMF)

Use `interpretacao_nmf_df` para redigir os resultados da Parte 1:

1. Palavras-chave de cada topico.
2. Quantidade de trabalhos por topico e distribuicao por ano.
3. Titulos representativos.
4. Interpretacao semantica final do topico.


## Parte 2 - Metodologia do artigo-base (BERTopic)

Aplicacao da estrategia do artigo `s41598-025-92190-7` no corpus BRACIS:

- embeddings contextuais com `all-MiniLM-L6-v2`;
- reducao de dimensionalidade com UMAP (`n_neighbors=15`, `random_state=100`);
- clusterizacao com HDBSCAN;
- representacao de topicos com c-TF-IDF (`ngram_range=(1,2)`, `top_n_words=20`);
- atribuicao probabilistica com `calculate_probabilities=True`;
- avaliacao por coerencia, separacao intertopicos e taxa de outliers.


Dependencias da Parte 2 (rode apenas se ainda nao instalou):

```python
%pip install bertopic sentence-transformers umap-learn hdbscan gensim
```


In [ ]:
import warnings

from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN

warnings.filterwarnings("ignore")


In [ ]:
# Reutiliza o corpus preparado na Parte 1.
docs_part2 = df_nlp["texto"].tolist()

# Stopwords base + termos genericos citados no artigo-base.
stopwords_part2 = set(ENGLISH_STOP_WORDS).union({"use", "add", "related"})

def tokenize_simple(texto):
    tokens = re.findall(r"[a-zA-Z][a-zA-Z\-]{1,}", str(texto).lower())
    return [t for t in tokens if t not in stopwords_part2 and len(t) >= 3]

tokenized_docs_part2 = [tokenize_simple(doc) for doc in docs_part2]
dictionary_part2 = Dictionary(tokenized_docs_part2)

print(f"Documentos para Parte 2: {len(docs_part2)}")


In [ ]:
def extrair_topic_terms_bertopic(topic_model, n_top_words=20):
    """Extrai as palavras mais relevantes de cada topico (exceto outlier -1)."""
    termos = []
    for topic_id in sorted(topic_model.get_topics().keys()):
        if topic_id == -1:
            continue
        palavras = topic_model.get_topic(topic_id) or []
        termos_topic = [w for w, _ in palavras[:n_top_words]]
        if termos_topic:
            termos.append(termos_topic)
    return termos

def coherence_cv(topic_terms, tokenized_docs, dictionary):
    topic_terms = [t for t in topic_terms if t]
    if not topic_terms:
        return float("nan")

    cm = CoherenceModel(
        topics=topic_terms,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence="c_v",
    )
    return float(cm.get_coherence())

def intertopic_distance_min(topic_model):
    info = topic_model.get_topic_info()
    topicos_validos = info[info["Topic"] != -1]["Topic"].tolist()
    if len(topicos_validos) < 2:
        return float("nan")

    # Distancia minima entre centroides de topicos no espaco de embeddings.
    embeddings = topic_model.topic_embeddings_[topicos_validos]
    sim = cosine_similarity(embeddings)
    dist = 1 - sim
    np.fill_diagonal(dist, np.nan)
    return float(np.nanmin(dist))

def topic_diversity(topic_terms, topk=10):
    cortes = [termos[:topk] for termos in topic_terms if termos]
    if not cortes:
        return float("nan")
    total = sum(len(x) for x in cortes)
    unicos = len({w for linha in cortes for w in linha})
    return float(unicos / total) if total else float("nan")


In [ ]:
embedding_model_name = "all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(embedding_model_name)

# Embeddings normalizados para melhorar comparacoes por similaridade.
embeddings_part2 = embedding_model.encode(
    docs_part2,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print("Embeddings gerados:", embeddings_part2.shape)


In [ ]:
avaliacoes_part2 = []
modelos_part2 = {}

# Tuning de k alvo entre 4 e 16, conforme intervalo metodologico do artigo-base.
for k in range(4, 17):
    umap_model = UMAP(n_neighbors=15, metric="cosine", random_state=100)
    hdbscan_model = HDBSCAN(
        min_cluster_size=20,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,
    )
    vectorizer_model = CountVectorizer(
        stop_words=list(stopwords_part2),
        ngram_range=(1, 2),
    )

    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        top_n_words=20,
        min_topic_size=20,
        nr_topics=k,
        calculate_probabilities=True,
        verbose=False,
    )

    topics, probs = topic_model.fit_transform(docs_part2, embeddings_part2)

    info = topic_model.get_topic_info()
    n_topicos_validos = int((info["Topic"] != -1).sum())
    taxa_outlier = float((np.array(topics) == -1).mean())

    termos = extrair_topic_terms_bertopic(topic_model, n_top_words=20)
    coerencia_cv_k = coherence_cv(termos, tokenized_docs_part2, dictionary_part2)
    dist_min = intertopic_distance_min(topic_model)
    diversidade_10 = topic_diversity(termos, topk=10)

    avaliacoes_part2.append(
        {
            "k_alvo": k,
            "topicos_encontrados": n_topicos_validos,
            "coerencia_c_v": coerencia_cv_k,
            "distancia_intertopico_min": dist_min,
            "diversidade_top10": diversidade_10,
            "taxa_outlier": taxa_outlier,
        }
    )

    # Guarda o modelo para reutilizar sem novo fit.
    modelos_part2[k] = (topic_model, topics, probs)

avaliacoes_part2_df = pd.DataFrame(avaliacoes_part2).sort_values(
    ["coerencia_c_v", "distancia_intertopico_min", "diversidade_top10"],
    ascending=[False, False, False],
).reset_index(drop=True)

avaliacoes_part2_df


In [ ]:
# O artigo-base reporta 5 topicos; mantemos k=5 para reproducao metodologica.
k_escolhido_part2 = 5
topic_model_part2, topics_part2, probs_part2 = modelos_part2[k_escolhido_part2]

doc_info_part2 = topic_model_part2.get_document_info(docs_part2)

df_nlp["topico_bertopic"] = doc_info_part2["Topic"].values
df_nlp["score_topico_bertopic"] = doc_info_part2["Probability"].fillna(0).values

topic_info_part2 = topic_model_part2.get_topic_info().copy()
topic_info_part2


In [ ]:
def tabela_topicos_bertopic(topic_model, top_n_words=12):
    linhas = []
    info = topic_model.get_topic_info()

    for _, row in info.iterrows():
        topic_id = int(row["Topic"])
        if topic_id == -1:
            continue

        palavras = topic_model.get_topic(topic_id) or []
        palavras_top = [w for w, _ in palavras[:top_n_words]]

        linhas.append(
            {
                "topico": topic_id,
                "qtd_documentos": int(row["Count"]),
                "palavras_chave": ", ".join(palavras_top),
            }
        )

    return pd.DataFrame(linhas).sort_values("qtd_documentos", ascending=False)

topicos_part2_df = tabela_topicos_bertopic(topic_model_part2, top_n_words=12)
topicos_part2_df


In [ ]:
def interpretar_topicos_bertopic(df_docs, topicos_df, top_n_titulos=3):
    linhas = []

    for _, linha in topicos_df.iterrows():
        t = int(linha["topico"])
        sub = df_docs[df_docs["topico_bertopic"] == t].copy()

        qtd_docs = len(sub)
        por_ano = sub["Ano"].value_counts().sort_index().to_dict()
        # Ordena por probabilidade de pertencer ao topico.
        titulos_rep = (
            sub.sort_values("score_topico_bertopic", ascending=False)["Titulo"]
            .head(top_n_titulos)
            .tolist()
        )

        termos = linha["palavras_chave"].split(", ")[:4]
        interpretacao = (
            f"Topico {t} foca em {', '.join(termos)}. "
            f"Documentos: {qtd_docs}. Distribuicao por ano: {por_ano}."
        )

        linhas.append(
            {
                "topico": t,
                "palavras_chave": linha["palavras_chave"],
                "qtd_documentos": qtd_docs,
                "distribuicao_anos": por_ano,
                "titulos_representativos": " | ".join(titulos_rep),
                "interpretacao_inicial": interpretacao,
            }
        )

    return pd.DataFrame(linhas).sort_values("qtd_documentos", ascending=False)

interpretacao_part2_df = interpretar_topicos_bertopic(df_nlp, topicos_part2_df)
interpretacao_part2_df


### Visualizacao UMAP 2D dos topicos (Parte 2)

Projecao em 2D dos embeddings para verificar separacao visual dos clusters gerados pelo BERTopic/HDBSCAN.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

umap_vis_model = UMAP(
    n_neighbors=15,
    n_components=2,
    metric="cosine",
    random_state=100,
)

# Projeta os embeddings para visualizacao.
emb_2d_part2 = umap_vis_model.fit_transform(embeddings_part2)

plot_df = pd.DataFrame({
    "UMAP-1": emb_2d_part2[:, 0],
    "UMAP-2": emb_2d_part2[:, 1],
    "topico": df_nlp["topico_bertopic"].values,
})

plot_df["topico_label"] = plot_df["topico"].apply(
    lambda t: "Outlier (-1)" if int(t) == -1 else f"Topico {int(t)}"
)

ordem = sorted(plot_df["topico"].unique(), key=lambda x: (x == -1, x))
ordem_labels = ["Outlier (-1)" if int(t) == -1 else f"Topico {int(t)}" for t in ordem]

plt.figure(figsize=(11, 8))
sns.scatterplot(
    data=plot_df,
    x="UMAP-1",
    y="UMAP-2",
    hue="topico_label",
    hue_order=ordem_labels,
    palette="tab20",
    s=30,
    alpha=0.8,
    linewidth=0,
)
plt.title("UMAP 2D dos documentos BRACIS (colorido por topico BERTopic)")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.legend(title="Topicos", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## Parte 5 - Comparacao entre NMF e BERTopic

Comparacao final entre:

- Parte 1: BoW/TF-IDF + NMF (baseline classico);
- Parte 2: BERTopic (embeddings + UMAP + HDBSCAN + c-TF-IDF).


In [ ]:
def topic_terms_from_df(topics_df_col):
    return [str(v).split(", ") for v in topics_df_col.tolist() if str(v).strip()]

# Parte 1 (NMF)
nmf_terms = topic_terms_from_df(nmf_topics_df["palavras_chave"])
coerencia_nmf = coherence_cv(nmf_terms, tokenized_docs_part2, dictionary_part2)
diversidade_nmf = topic_diversity(nmf_terms, topk=10)

# Parte 2 (BERTopic)
bertopic_terms = extrair_topic_terms_bertopic(topic_model_part2, n_top_words=20)
coerencia_bertopic = coherence_cv(bertopic_terms, tokenized_docs_part2, dictionary_part2)
diversidade_bertopic = topic_diversity(bertopic_terms, topk=10)
taxa_outlier_part2 = float((df_nlp["topico_bertopic"] == -1).mean())

# Tabela consolidada para comparacao objetiva dos metodos.
comparacao_df = pd.DataFrame(
    [
        {
            "metodo": "NMF (Parte 1)",
            "n_topicos": int(nmf_topics_df["topico"].nunique()),
            "coerencia_c_v": coerencia_nmf,
            "diversidade_top10": diversidade_nmf,
            "cobertura_docs": 1.0,
            "metrica_extra": f"erro_reconstrucao={nmf_model.reconstruction_err_:.4f}",
        },
        {
            "metodo": "BERTopic (Parte 2)",
            "n_topicos": int((topic_info_part2["Topic"] != -1).sum()),
            "coerencia_c_v": coerencia_bertopic,
            "diversidade_top10": diversidade_bertopic,
            "cobertura_docs": 1.0 - taxa_outlier_part2,
            "metrica_extra": f"outlier_rate={taxa_outlier_part2:.3f}",
        },
    ]
).sort_values("coerencia_c_v", ascending=False).reset_index(drop=True)

comparacao_df


### Graficos da comparacao

Visualizacoes para apoiar a discussao dos resultados:

- barras: coerencia, diversidade e cobertura por metodo;
- tuning BERTopic por k: coerencia, separacao minima e taxa de outliers.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# 1) Barras: comparacao direta entre metodos
metricas = ["coerencia_c_v", "diversidade_top10", "cobertura_docs"]
rotulos = {
    "coerencia_c_v": "Coerencia c_v",
    "diversidade_top10": "Diversidade (top-10)",
    "cobertura_docs": "Cobertura de documentos",
}

plot_comp = comparacao_df.copy()
plot_comp_long = plot_comp.melt(
    id_vars=["metodo"],
    value_vars=metricas,
    var_name="metrica",
    value_name="valor",
)
plot_comp_long["metrica"] = plot_comp_long["metrica"].map(rotulos)

plt.figure(figsize=(10, 5.5))
sns.barplot(
    data=plot_comp_long,
    x="metrica",
    y="valor",
    hue="metodo",
    palette="Set2",
)
plt.title("Comparacao de metodos: NMF vs BERTopic")
plt.xlabel("Metrica")
plt.ylabel("Valor")
plt.ylim(0, 1.05)
plt.legend(title="Metodo", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# 2) Tuning BERTopic
tuning_df = avaliacoes_part2_df.sort_values("k_alvo").copy()

def _ylim_with_margin(series, frac=0.12, min_span=1e-3):
    vals = pd.Series(series).dropna().astype(float)
    vmin, vmax = float(vals.min()), float(vals.max())
    span = vmax - vmin
    if span < min_span:
        pad = max(abs(vmin) * 0.03, min_span)
        return (vmin - pad, vmax + pad)
    pad = span * frac
    return (vmin - pad, vmax + pad)

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Coerencia
axes[0].plot(
    tuning_df["k_alvo"],
    tuning_df["coerencia_c_v"],
    marker="o",
    color="#1f77b4",
    linewidth=2,
)
axes[0].axvline(x=k_escolhido_part2, linestyle="--", color="#7f7f7f", linewidth=1.5)
axes[0].set_ylabel("Coerencia c_v")
axes[0].set_ylim(*_ylim_with_margin(tuning_df["coerencia_c_v"]))
axes[0].set_title("Coerencia por k")

# Distancia intertopico minima
axes[1].plot(
    tuning_df["k_alvo"],
    tuning_df["distancia_intertopico_min"],
    marker="^",
    color="#2ca02c",
    linewidth=2,
)
axes[1].axvline(x=k_escolhido_part2, linestyle="--", color="#7f7f7f", linewidth=1.5)
axes[1].set_ylabel("Dist. intertopico min")
axes[1].set_ylim(*_ylim_with_margin(tuning_df["distancia_intertopico_min"]))
axes[1].set_title("Separacao minima entre topicos por k")

# Taxa de outliers
axes[2].plot(
    tuning_df["k_alvo"],
    tuning_df["taxa_outlier"],
    marker="s",
    color="#d62728",
    linewidth=2,
)
axes[2].axvline(
    x=k_escolhido_part2,
    linestyle="--",
    color="#7f7f7f",
    linewidth=1.5,
    label=f"k escolhido = {k_escolhido_part2}",
)
axes[2].set_ylabel("Taxa de outliers")
axes[2].set_ylim(*_ylim_with_margin(tuning_df["taxa_outlier"]))
axes[2].set_title("Outliers por k")
axes[2].legend(loc="upper left", frameon=True)

for ax in axes:
    ax.grid(True, alpha=0.35)

axes[2].set_xlabel("k alvo (nr_topics)")
fig.suptitle("Tuning BERTopic: coerencia, separacao e outliers por k", y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

# Nota quando as curvas variam pouco.
quase_constantes = {
    "coerencia_c_v": tuning_df["coerencia_c_v"].nunique(),
    "distancia_intertopico_min": tuning_df["distancia_intertopico_min"].nunique(),
    "taxa_outlier": tuning_df["taxa_outlier"].nunique(),
}
if all(v <= 2 for v in quase_constantes.values()):
    print(
        "Observacao: as metricas de tuning variaram pouco entre os valores de k. "
        "Isso indica estabilidade do modelo para este corpus."
    )


In [ ]:
melhor = comparacao_df.iloc[0]

print("Resumo comparativo:")
print(f"- Melhor coerencia c_v: {melhor['metodo']} ({melhor['coerencia_c_v']:.4f})")
print(
    f"- BERTopic: topicos={int((topic_info_part2['Topic'] != -1).sum())}, cobertura={(1.0 - taxa_outlier_part2):.2%}, outliers={taxa_outlier_part2:.2%}"
)
print(
    "- Interpretacao: NMF fornece baseline lexical simples; BERTopic separa melhor semantica com embeddings."
)
